# Lecture 8 — Exercise 5: Vectorization with Multiple Variables

Adapted from the Machine Learning Specialization labs by Andrew Ng (DeepLearning.AI and Stanford Online).

In Exercise 5 of Lecture 7 you wrote the model, the cost and the gradient without loops for one feature. Here you do the same with $n$ features. In the Exercise 1-4 notebook, `MI-L08-multiple-linear-regression`, the cost and the gradient loop over the houses one at a time. In this notebook you replace those loops with NumPy operations that work on the whole matrix $\mathbf{X}$ at once. This is called vectorization. The code gets shorter, and it runs much faster on large datasets because NumPy does the loops in compiled code. Work on it at home after the lecture, before you open the Kaggle notebook. The Kaggle notebook uses your three functions.

You write three functions.

- `compute_model_output` in Exercise 5a
- `compute_cost` in Exercise 5b
- `compute_gradient` in Exercise 5c

Before you start, save your own copy with File > Save a copy in Drive.

Complete the lines marked `# TODO` and run the cells from top to bottom. After each exercise, the cell marked `# Check your result` compares your function with the loop version and prints `Check passed` when both give the same numbers. At the end you time both versions on a large dataset and run gradient descent with your functions.

## 1 - Setup

In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)  # print NumPy arrays with two decimals

In [ ]:
def load_house_data():
    """
    Loads houses.txt, with 100 houses. Columns 0 to 3 are the size in sqft, the number of
    bedrooms, the number of floors and the age in years. Column 4 is the price in $1,000.
    The file has no header line, so all 100 houses are read.
    It tries the copy in the course repository on GitHub first and then the file
    data/houses.txt in the folder where the notebook runs. In Colab that folder is /content.

    Returns:
      X (ndarray (m,4)): size, bedrooms, floors and age of each house
      y (ndarray (m,)) : price of each house
    """
    url = "https://raw.githubusercontent.com/andresmasegosa/MI_E26/main/lecture08/data/houses.txt"
    local_file = "data/houses.txt"
    try:
        data = np.loadtxt(url, delimiter=',')
        source = "GitHub"
    except Exception:
        try:
            data = np.loadtxt(local_file, delimiter=',')
            source = local_file
        except Exception:
            raise FileNotFoundError(
                "Could not load houses.txt from GitHub or from data/houses.txt. "
                "Check your internet connection, or put houses.txt in a folder called data "
                "in the folder where the notebook runs. In Colab, upload it to /content/data.") from None
    print(f"Loaded {data.shape[0]} houses from {source}")

    X = data[:,:4]
    y = data[:,4]

    # One house has 0 bedrooms. Set it to 1 so that size/bedrooms is defined.
    X[X[:,1]==0,1]=1

    return X, y

In [ ]:
# load the dataset
X_train, y_train = load_house_data()
m, n = X_train.shape
print(f"X Shape: {X_train.shape}, y Shape: {y_train.shape}")

# arbitrary parameters, used to test the functions below
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])

The dataset has $m = 100$ houses and $n = 4$ features (size, bedrooms, floors and age). The values of `w_init` and `b_init` are arbitrary. We only use them to compare the vectorized functions with the loop versions.

## 2 - The loop versions

These are the functions from the Exercise 1-4 notebook, renamed with `_loop` at the end. Your vectorized functions must give the same results. The checks compare with `np.isclose` and `np.allclose`, which accept tiny rounding differences between the two versions.

In [ ]:
def compute_cost_loop(X, y, w, b):
    """
    compute cost, with a loop over the examples
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f_wb_i = np.dot(X[i], w) + b           # the dot product of two (n,) vectors is a number
        cost = cost + (f_wb_i - y[i])**2
    cost = cost / (2 * m)
    return cost

In [ ]:
def compute_gradient_loop(X, y, w, b):
    """
    Computes the gradient for linear regression, with loops over the examples and the features
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      dj_db (scalar):       The partial derivative of the cost with respect to b
      dj_dw (ndarray (n,)): The partial derivatives of the cost with respect to the weights in w
    """
    m,n = X.shape           #(number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err * X[i, j]
        dj_db = dj_db + err
    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_db, dj_dw

## 3 - Exercise 5: Vectorization

### Exercise 5a: Model

For one example the model is

$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b = \sum_{j=1}^{n} w_j x^{(i)}_j + b $$

For all $m$ examples at once it is a matrix-vector product.

$$ \mathbf{f} = \mathbf{X}\mathbf{w} + b $$

$\mathbf{X}$ has shape $(m, n)$ and $\mathbf{w}$ has shape $(n,)$, so $\mathbf{X}\mathbf{w}$ has shape $(m,)$, one prediction per example. NumPy adds the scalar $b$ to every entry. In NumPy the product is `X @ w`.

In the formulas we count from 1, as in the slides. Python counts from 0, so $\mathbf{x}^{(i)}$ is `X[i-1]` and $w_j$ is `w[j-1]`.

In the Exercise 1-4 notebook `compute_model_output` loops over the rows of `X`. Complete it here so that it returns the predictions for all the rows of `X`, without a loop.

In [ ]:
def compute_model_output(X, w, b):
    """
    Predictions of the linear model for all the examples
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      f_wb (ndarray (m,)): prediction for every example
    """
    f_wb = None  # TODO: the predictions for all the rows of X, without a loop

    return f_wb

In [ ]:
# Check your result
f_wb = compute_model_output(X_train, w_init, b_init)
assert f_wb is not None, "compute_model_output returns None. Complete the line marked TODO first."
f_wb_loop = np.array([np.dot(X_train[i], w_init) + b_init for i in range(m)])
assert np.shape(f_wb) == (m,), f"compute_model_output should return shape ({m},), not {np.shape(f_wb)}"
assert np.allclose(f_wb, f_wb_loop), "compute_model_output does not match the loop version"
X_small = np.array([[1., 2.], [2., 0.], [0., 1.]])
try:
    f_small = compute_model_output(X_small, np.array([1., 1.]), 1.)
except ValueError:
    f_small = None
assert f_small is not None and np.allclose(f_small, [4., 3., 2.]), "compute_model_output gives the wrong values on a small example. Use the arguments X, w and b of the function, not X_train or w_init."
print(f"First three predictions: {f_wb[:3]}")
print("Check passed")

Expected output:

    First three predictions: [-575.51 -416.06 1047.49]
    Check passed

The last check uses a small example that you can do by hand. With the three rows $\mathbf{x}^{(1)} = (1, 2)$, $\mathbf{x}^{(2)} = (2, 0)$ and $\mathbf{x}^{(3)} = (0, 1)$, $\mathbf{w} = (1, 1)$ and $b = 1$, the predictions are 4, 3 and 2.

### Exercise 5b: Cost

The cost is

$$J(\mathbf{w},b) = \frac{1}{2m} \sum_{i=1}^{m} \left(f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

Put the errors of all the examples in one vector $\mathbf{e} = \mathbf{X}\mathbf{w} + b - \mathbf{y}$ of shape $(m,)$. The sum of their squares is the dot product $\mathbf{e} \cdot \mathbf{e}$, so

$$J(\mathbf{w},b) = \frac{1}{2m}\, \mathbf{e} \cdot \mathbf{e}$$

Complete `compute_cost` without a loop. You can call your `compute_model_output`.

Keep `y` as a 1-D array of shape $(m,)$. If `y` had shape $(m, 1)$, subtracting it from an array of shape $(m,)$ would give an $(m, m)$ matrix without any error message, and the cost would be wrong.

In [ ]:
def compute_cost(X, y, w, b):
    """
    compute cost, vectorized
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    err = None   # TODO: the error vector e, shape (m,)
    cost = None  # TODO: the cost J, a scalar

    return cost

In [ ]:
# Check your result
cost = compute_cost(X_train, y_train, w_init, b_init)
assert cost is not None, "compute_cost returns None. Complete the lines marked TODO first."
assert np.ndim(cost) == 0, f"compute_cost returns an array of shape {np.shape(cost)}, but J(w,b) is one number."
assert np.isclose(cost, compute_cost_loop(X_train, y_train, w_init, b_init)), "compute_cost does not match the loop version"
X_small = np.array([[1., 2.], [2., 0.], [0., 1.]])
y_small = np.array([2., 2., 2.])
try:
    cost_small = compute_cost(X_small, y_small, np.array([1., 1.]), 1.)
except ValueError:
    cost_small = None
assert cost_small is not None and np.isclose(cost_small, 5/6), "compute_cost gives the wrong value on a small example. Use the arguments X, y, w and b of the function, not X_train or y_train."
print(f"Cost at w_init, b_init: {cost:0.2f}")
print("Check passed")

Expected output:

    Cost at w_init, b_init: 223658.99
    Check passed

The last check uses the small example of Exercise 5a with $\mathbf{y} = (2, 2, 2)$. The errors are 2, 1 and 0, so $J(\mathbf{w},b) = \frac{1}{2 \cdot 3}(4 + 1 + 0) = \frac{5}{6}$.

### Exercise 5c: Gradient

The partial derivatives are

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})
\end{align}
$$

With the error vector $\mathbf{e}$ from Exercise 5b, all $n$ partial derivatives with respect to the weights come from one matrix-vector product.

$$\nabla_{\mathbf{w}} J = \frac{1}{m}\, \mathbf{X}^T \mathbf{e}$$

$\mathbf{X}^T$ has shape $(n, m)$ and $\mathbf{e}$ has shape $(m,)$, so the result has shape $(n,)$, one entry per weight. The partial derivative with respect to $b$ is the mean of the errors. In NumPy $\mathbf{X}^T$ is `X.T`.

As in the Exercise 1-4 notebook, `compute_gradient` returns `dj_db` first and then `dj_dw`. The Lecture 7 notebooks returned them in the other order. Complete `compute_gradient` without a loop.

In [ ]:
def compute_gradient(X, y, w, b):
    """
    Computes the gradient for linear regression, vectorized
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters
      b (scalar)       : model parameter

    Returns:
      dj_db (scalar):       The partial derivative of the cost with respect to b
      dj_dw (ndarray (n,)): The partial derivatives of the cost with respect to the weights in w
    """
    m = X.shape[0]
    err = None    # TODO: the error vector e, shape (m,)
    dj_db = None  # TODO: the partial derivative with respect to b, a scalar
    dj_dw = None  # TODO: the partial derivatives with respect to w, shape (n,)

    return dj_db, dj_dw

In [ ]:
# Check your result
dj_db, dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
assert dj_db is not None and dj_dw is not None, "compute_gradient returns None. Complete the lines marked TODO first."
assert np.ndim(dj_db) == 0, "dj_db should be one number, the mean of the errors. Check that compute_gradient returns dj_db first."
dj_db_loop, dj_dw_loop = compute_gradient_loop(X_train, y_train, w_init, b_init)
assert np.shape(dj_dw) == (n,), f"dj_dw should have shape ({n},), not {np.shape(dj_dw)}"
assert np.isclose(dj_db, dj_db_loop), "dj_db does not match the loop version"
assert np.allclose(dj_dw, dj_dw_loop), "dj_dw does not match the loop version"
X_small = np.array([[1., 2.], [2., 0.], [0., 1.]])
y_small = np.array([2., 2., 2.])
try:
    tmp_dj_db, tmp_dj_dw = compute_gradient(X_small, y_small, np.array([1., 1.]), 1.)
except ValueError:
    tmp_dj_db, tmp_dj_dw = None, None
assert tmp_dj_db is not None and np.isclose(tmp_dj_db, 1.) and np.allclose(tmp_dj_dw, [4/3, 4/3]), "compute_gradient gives the wrong values on a small example. Use the arguments X, y, w and b of the function, not X_train or y_train."
print(f"dj_db at w_init, b_init: {dj_db:0.2f}")
print(f"dj_dw at w_init, b_init: {dj_dw}")
print("Check passed")

Expected output:

    dj_db at w_init, b_init: -67.63
    dj_dw at w_init, b_init: [ 6.45e+03 -1.15e+02 -6.80e+00 -1.96e+04]
    Check passed

The last check uses the small example of Exercise 5b, where $\mathbf{e} = (2, 1, 0)$. The partial derivative with respect to $b$ is $\frac{1}{3}(2 + 1 + 0) = 1$, and $\nabla_{\mathbf{w}} J = \frac{1}{3}\mathbf{X}^T\mathbf{e} = (\frac{4}{3}, \frac{4}{3})$.

## 4 - How much faster is it?

On 100 houses both versions are fast. The difference shows on larger data. The next cell builds a synthetic dataset with $m = 100{,}000$ examples and 4 features. The cell after it times one gradient computation with each version. `%timeit` runs the code several times and reports the average. The loop version is slow, so `%timeit` runs it only three times. `-n` and `-r` set how many times each version runs, and `-o` keeps the result, so that the last line can divide the two average times.

In [ ]:
# a synthetic dataset with 100,000 examples and 4 features
rng = np.random.default_rng(0)
# rng.normal draws numbers from a normal distribution with mean 0 and standard deviation 1
X_big = rng.normal(size=(100_000, 4))
y_big = rng.normal(size=100_000)
w_big = rng.normal(size=4)
b_big = 0.5

In [ ]:
# stop here with a clear message if Exercise 5c is not done yet
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
assert tmp_dj_db is not None and tmp_dj_dw is not None, "Finish Exercise 5c first."

t_loop = %timeit -o -n 1 -r 3 compute_gradient_loop(X_big, y_big, w_big, b_big)
t_vec = %timeit -o -n 10 -r 3 compute_gradient(X_big, y_big, w_big, b_big)
print(f"The vectorized gradient is about {t_loop.average / t_vec.average:,.0f} times faster.")

In our run the vectorized gradient was a few hundred times faster. Your number will be different, because it depends on the computer.

## 5 - Gradient descent with the vectorized functions

The next two cells copy `zscore_normalize_features` and `gradient_descent` from the Exercise 1-4 notebook without changes. `gradient_descent` calls the cost and gradient functions it receives as arguments. The third cell passes your vectorized `compute_cost` and `compute_gradient` and runs it on the scaled features, with the settings of Section 6.3 of the Exercise 1-4 notebook. The result should be the same, a cost of 231.05.

In [ ]:
def zscore_normalize_features(X):
    """
    Scales each column of X with z-score normalization

    Args:
      X (ndarray (m,n))     : input data, m examples, n features

    Returns:
      X_norm (ndarray (m,n)): X with every column scaled
      X_mean (ndarray (n,)) : mean of each feature
      X_sigma (ndarray (n,)): standard deviation of each feature
    """
    # find the mean of each column/feature
    X_mean     = np.mean(X, axis=0)                 # X_mean has shape (n,)
    # find the standard deviation of each column/feature
    X_sigma  = np.std(X, axis=0)                  # X_sigma has shape (n,)
    # NumPy subtracts X_mean from every row of X and divides every row by X_sigma. This is called broadcasting.
    X_norm = (X - X_mean) / X_sigma

    return (X_norm, X_mean, X_sigma)

In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking
    num_iters gradient steps with learning rate alpha

    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent

    Returns:
      w (ndarray (n,))    : Updated values of parameters
      b (scalar)          : Updated value of parameter
      J_history (list)    : cost after each iteration
      """

    # A list to store the cost J at each iteration, for the plots later
    J_history = []
    w = copy.deepcopy(w_in)
    b = b_in

    for i in range(num_iters):

        # Calculate the gradient at the current w and b
        dj_db, dj_dw = gradient_function(X, y, w, b)

        # Update the parameters using alpha and the gradient
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion
            J_history.append( cost_function(X, y, w, b))

        # Print the cost 10 times during the run, or at every iteration if num_iters < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")

    return w, b, J_history

In [ ]:
# stop here with a clear message if Exercises 5b and 5c are not done yet
assert compute_cost(X_train, y_train, w_init, b_init) is not None, "Finish Exercise 5b first."
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
assert tmp_dj_db is not None and tmp_dj_dw is not None, "Finish Exercise 5c first."

X_norm, X_mean, X_sigma = zscore_normalize_features(X_train)
w_final, b_final, J_hist = gradient_descent(X_norm, y_train, np.zeros(n), 0.,
                                            compute_cost, compute_gradient, 0.1, 1000)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
print(f"Cost after 1000 iterations: {J_hist[-1]:0.2f}")

Expected output:

    Iteration    0: Cost 57299.69   
    Iteration  100: Cost   234.01   
    Iteration  200: Cost   231.06   
    Iteration  300: Cost   231.05   
    Iteration  400: Cost   231.05   
    Iteration  500: Cost   231.05   
    Iteration  600: Cost   231.05   
    Iteration  700: Cost   231.05   
    Iteration  800: Cost   231.05   
    Iteration  900: Cost   231.05   
    b,w found by gradient descent: 362.24,[111.63 -21.35 -33.16 -37.92] 
    Cost after 1000 iterations: 231.05

## 6 - Optional: degree 20 with more iterations

This part is optional. Do it at home, after Exercise 3. With the vectorized functions 100,000 iterations take a few seconds. Run the cells below for degree 20 and compare the cost after 100,000 iterations with the minimum cost and with your table of Exercise 3.

In [ ]:
def poly_features(X, degree):
    """
    Adds the columns size^2, ..., size^degree to X. The size is column 0 of X.

    Args:
      X (ndarray (m,n)): Data, m examples with n features
      degree (int)     : highest power of the size

    Returns:
      X_poly (ndarray (m,n+degree-1)): X followed by the new columns
    """
    new_columns = [X[:, 0]**d for d in range(2, degree + 1)]
    return np.column_stack([X] + new_columns)


def min_cost(X, y):
    """
    Lowest cost J(w,b) that any w, b can reach on X, y. np.linalg.lstsq finds
    the best w and b directly, without gradient descent.

    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values

    Returns:
      cost (scalar): the minimum of J(w,b)
    """
    A = np.column_stack([X, np.ones(X.shape[0])])   # a column of ones for b
    params = np.linalg.lstsq(A, y, rcond=None)[0]
    return compute_cost(X, y, params[:-1], params[-1])

In [ ]:
# stop here with a clear message if Exercises 5b and 5c are not done yet
assert compute_cost(X_train, y_train, w_init, b_init) is not None, "Finish Exercise 5b first."
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
assert tmp_dj_db is not None and tmp_dj_dw is not None, "Finish Exercise 5c first."

degree = 20
X_scaled, _, _ = zscore_normalize_features(X_train)
X_poly, _, _ = zscore_normalize_features(poly_features(X_scaled, degree))

w_final, b_final, J_hist = gradient_descent(X_poly, y_train, np.zeros(X_poly.shape[1]), 0.,
                                            compute_cost, compute_gradient, 0.1, 100000)
print(f"Degree {degree}: cost after 100000 iterations {J_hist[-1]:0.2f}, minimum cost {min_cost(X_poly, y_train):0.2f}")

Expected output:

    Iteration    0: Cost 57376.52   
    Iteration 10000: Cost   210.04   
    Iteration 20000: Cost   204.59   
    Iteration 30000: Cost   200.39   
    Iteration 40000: Cost   197.05   
    Iteration 50000: Cost   194.40   
    Iteration 60000: Cost   192.29   
    Iteration 70000: Cost   190.62   
    Iteration 80000: Cost   189.28   
    Iteration 90000: Cost   188.22   
    Degree 20: cost after 100000 iterations 187.37, minimum cost 147.84